# BBBC031: training Stochastic Mixture (NB=3 e NB=7) su Colab

Questo notebook addestra **Extended Stochastic Mixture NCA** su **5 immagini** BBBC031 per due dimensioni di vicinato (3 e 7), adatto a Google Colab.

**Prima di eseguire:**
1. **GPU (consigliata):** Menu *Runtime → Cambia tipo di runtime* e seleziona **GPU**. Se dopo `pip install -r requirements.txt` vedi "Torch not compiled with CUDA", riavvia il runtime e **non** rieseguire la cella di pip install (usa il PyTorch già presente su Colab), oppure reinstalla torch con CUDA.
2. Carica il dataset BBBC031 (cartella `BBBC031_v1_dataset` con sottocartella `Images/`) e il CSV ground truth su Google Drive, oppure caricali nella sessione Colab.
3. Imposta sotto i path `DATASET_DIR` e `CSV_PATH` (e opzionalmente `REPO_URL` se il repo è su GitHub).

## 1. Setup: clone repo e dipendenze

In [ ]:
# Clone del repo (cambia REPO_URL con il tuo repo se necessario)
REPO_URL = "https://github.com/luigidaddario/MNCA.git"  # oppure il path del tuo fork

!git clone --depth 1 {REPO_URL} /content/MNCA
%cd /content/MNCA
!pip install -q -r requirements.txt

## 2. Path dei dati (Drive o upload)

Scegli **una** delle due opzioni: monta Drive e imposta i path, oppure usa i path dove hai caricato i file nella sessione.

**Tutti gli output** (modelli, loss, figure, video) vengono salvati su Drive in `DRIVE_OUTPUT_DIR`, così restano disponibili anche se il runtime si scollega.


In [ ]:
# Opzione A: monta Google Drive e imposta i path alla cartella BBBC031
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = "/content/drive/MyDrive/BBBC031_v1_dataset"   # cartella con Images/ e Masks/
CSV_PATH    = "/content/drive/MyDrive/BBBC031_v1_DatasetGroundTruth.csv"  # CSV ground truth (sep=";")

# Opzione B: se hai caricato i file in /content/ (es. zip estratto):
# DATASET_DIR = "/content/BBBC031_v1_dataset"
# CSV_PATH    = "/content/BBBC031_v1_DatasetGroundTruth.csv"

import os
assert os.path.isdir(DATASET_DIR), f"Dataset non trovato: {DATASET_DIR}"
assert os.path.isfile(CSV_PATH), f"CSV non trovato: {CSV_PATH}"
assert os.path.isdir(os.path.join(DATASET_DIR, "Images")), "Manca la sottocartella Images/"
print("Path dati OK.")

# Cartella su Drive dove salvare modelli, figure e video (persistono se il runtime si scollega)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MNCA_bbbc031_outputs"
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031"), exist_ok=True)
print("Output su Drive:", DRIVE_OUTPUT_DIR)

In [ ]:
# Select 5 images that have a cell-mask image available
import pandas as pd
images_dir = os.path.join(DATASET_DIR, "Images")
df_gt = pd.read_csv(CSV_PATH, sep=";")
all_names = df_gt["ImageName"].unique()
# Only images that have a cell-mask image (dataset naming convention)
available = [n for n in all_names if os.path.isfile(os.path.join(images_dir, f"{n}_CELLMASK.png"))]
EXAMPLE_IMAGES = available[:5]
assert len(EXAMPLE_IMAGES) >= 5, f"At least 5 images with cell mask required; found {len(EXAMPLE_IMAGES)}."
print("Training on 5 images:", EXAMPLE_IMAGES)

## 3. Training: Stochastic Mixture on **5 images** (NB=3 and NB=7)

**NB=3** uses the default hyperparameters (learning rate \(10^{-3}\), 4000 steps).  
**NB=7** uses a **reduced learning rate** (\(3\times10^{-4}\)) and **6000 steps** to avoid divergence and allow stable convergence (see thesis Implementation chapter, BBBC031 section).

In [ ]:
# In-process execution so the progress bar (tqdm) is visible in the notebook.
# NB=3: default LR 1e-3, 4000 steps.
# NB=7: reduced LR 3e-4 and 6000 steps to avoid divergence (see thesis, Implementation § BBBC031).
import sys
import os
os.chdir("/content/MNCA")
if "/content/MNCA" not in sys.path:
    sys.path.insert(0, "/content/MNCA")

from experiments.train_bbbc031_mnca import main

# Hyperparameters: for NB=7 we use a lower learning rate and more steps for stability.
TRAIN_CONFIG = {
    3: {"learning_rate": 1e-3, "total_steps": 4000, "milestones": [1500, 3000]},
    7: {"learning_rate": 3e-4, "total_steps": 6000, "milestones": [2500, 4500]},
}

for img_idx, example_image in enumerate(EXAMPLE_IMAGES):
    for nb in (3, 7):
        cfg = TRAIN_CONFIG[nb]
        ckpt = f"{DRIVE_OUTPUT_DIR}/models/bbbc031_stochastic_NB{nb}_img{img_idx}.pth"
        print(f"\n--- Image {img_idx+1}/5: {example_image} | NB={nb} (LR={cfg['learning_rate']}, steps={cfg['total_steps']}) ---")
        sys.argv = [
            "train_bbbc031_mnca.py",
            "--dataset_dir", DATASET_DIR,
            "--csv_path", CSV_PATH,
            "--example_image", example_image,
            "--stochastic", "--neighborhood_size", str(nb),
            "--checkpoint_path", ckpt,
            "--total_steps", str(cfg["total_steps"]),
            "--learning_rate", str(cfg["learning_rate"]),
            "--milestones", str(cfg["milestones"][0]), str(cfg["milestones"][1]),
        ]
        main()

# Copy comparison figures from training output to Drive
import shutil
repo_figs = "/content/MNCA/thesis-latex/figs/bbbc031"
drive_figs = os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031")
if os.path.isdir(repo_figs):
    for f in os.listdir(repo_figs):
        src = os.path.join(repo_figs, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(drive_figs, f))
    print("Training figures copied to Drive.")

*(Il training NB=3 e NB=7 per tutte e 5 le immagini è eseguito nella sezione 3.)*

In [ ]:
# Training già eseguito nella cella sopra (5 immagini × NB=3 e NB=7).
pass

## 5. (Optional) Comparison figures and download

Figures are saved during training. Below we generate and display the comparison figures (ground truth vs model) for NB=3 and NB=7.

In [ ]:
import subprocess
import shutil
os.chdir("/content/MNCA")
fig_dir = os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031")
for img_idx, example_image in enumerate(EXAMPLE_IMAGES):
    for nb in (3, 7):
        ckpt = f"{DRIVE_OUTPUT_DIR}/models/bbbc031_stochastic_NB{nb}_img{img_idx}.pth"
        subprocess.run([
            "python", "experiments/bbbc031_mnca_demo.py",
            "--dataset_dir", DATASET_DIR, "--csv_path", CSV_PATH,
            "--example_image", example_image,
            "--checkpoint", ckpt, "--neighborhood_size", str(nb), "--stochastic",
            "--out_dir", fig_dir, "--num_steps", "20",
        ], check=True)
        src = os.path.join(fig_dir, f"bbbc031_gt_vs_mnca_NB{nb}_stochastic.png")
        dst = os.path.join(fig_dir, f"bbbc031_gt_vs_mnca_NB{nb}_img{img_idx}.png")
        if os.path.isfile(src):
            shutil.copy(src, dst)

In [ ]:
from IPython.display import Image, display

fig_dir = os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031")
import glob
for path in sorted(glob.glob(os.path.join(fig_dir, "bbbc031_gt_vs_mnca_*.png"))):
    name = os.path.basename(path)
    print(name)
    display(Image(path, width=500))


## 6. Video dell'evoluzione NCA

Genera un video che mostra l'evoluzione della maschera passo dopo passo (da seed alla predizione finale). Su Colab serve ffmpeg (di solito già presente); altrimenti viene salvato un GIF.

In [ ]:
# Video per la prima immagine, modello NB=3 (cambia esempio o checkpoint se vuoi)
import subprocess
os.chdir("/content/MNCA")
video_dir = os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031")
example_img = EXAMPLE_IMAGES[0]
ckpt = os.path.join(DRIVE_OUTPUT_DIR, "models", "bbbc031_stochastic_NB3_img0.pth")
subprocess.run([
    "python", "experiments/bbbc031_mnca_demo.py",
    "--dataset_dir", DATASET_DIR, "--csv_path", CSV_PATH,
    "--example_image", example_img, "--checkpoint", ckpt,
    "--neighborhood_size", "3", "--stochastic", "--num_steps", "20",
    "--out_dir", video_dir, "--save_video",
    "--video_path", os.path.join(video_dir, "bbbc031_evolution_NB3_img0.mp4"),
], check=True)

from IPython.display import Video
video_path = os.path.join(DRIVE_OUTPUT_DIR, "figs", "bbbc031", "bbbc031_evolution_NB3_img0.mp4")
if os.path.isfile(video_path):
    display(Video(video_path, width=400))
else:
    gif_path = video_path.replace(".mp4", ".gif")
    if os.path.isfile(gif_path):
        display(Image(gif_path, width=400))
    else:
        print("File video non trovato. Esegui prima il training e il demo con --save_video.")


In [ ]:
# Scarica i checkpoint, le loss, le figure e il video (zip)
!zip -r /content/bbbc031_outputs.zip $DRIVE_OUTPUT_DIR
from google.colab import files
files.download("/content/bbbc031_outputs.zip")
